# AIaaS — Combined Benchmark Report

One place to load every result JSON the suite produces and render a combined,
charted report across platforms and workloads. This notebook **reuses the package
scripts** rather than re-implementing them:
- `compare_results.py` → the serving / training / proxy comparison table
- `cost_model.py` → `$/M-tokens` (loaded if present; it ships in a separate PR)

It also summarizes the cross-framework, peak-ceiling, embeddings, and image-gen
result schemas, and renders a few bar charts.

> Run the individual benchmark notebooks first (each writes JSON into its
> `*_results/` folder), then run this from the `aiaas-benchmarking/` directory.

## 1. Setup — locate results and import the package scripts

In [ ]:
import os, sys, glob, json
sys.path.insert(0, os.getcwd())   # so compare_results / cost_model import

import compare_results as CR
try:
    import cost_model as CM
    HAVE_COST = True
except Exception as e:
    HAVE_COST = False
    print("cost_model not available (ships in a separate PR):", e)

# Where each benchmark writes its JSON. Adjust if you collected them elsewhere.
RESULT_GLOBS = [
    "vllm_bench_results/vllm_serving_*.json",
    "lora_train_results/lora_train_*.json",
    "optimum_bench_results/optimum_bench_*.json",
    "trtllm_bench_results/trtllm_bench_*.json",
    "embeddings_bench_results/embeddings_bench_*.json",
    "image_gen_bench_results/image_gen_bench_*.json",
]
paths = []
for g in RESULT_GLOBS:
    paths.extend(sorted(glob.glob(g)))
print(f"found {len(paths)} result files")

def load(paths):
    out = []
    for p in paths:
        try:
            d = json.load(open(p))
            if isinstance(d, dict):
                out.append((p, d))
        except Exception as e:
            print(f"skip {p}: {e}")
    return out
RUNS = load(paths)
by_schema = {}
for p, d in RUNS:
    by_schema.setdefault(d.get("schema", "?"), []).append((p, d))
print("schemas:", {k: len(v) for k, v in by_schema.items()})


## 2. Serving / training / proxy comparison (via `compare_results.py`)

In [ ]:
import pandas as pd

rows = {}
for p, run in RUNS:
    schema = run.get("schema", "")
    if schema.startswith("vllm-serving-bench"):
        rows[CR.label(run)] = CR.flatten_vllm(run)
    elif schema.startswith("lora-train-bench"):
        rows[CR.label(run)] = {**rows.get(CR.label(run), {}), **CR.flatten_train(run)}
    elif "tests" in run:
        rows[CR.label(run)] = {**rows.get(CR.label(run), {}), **CR.flatten_poc(run)}

if rows:
    cmp_df = pd.DataFrame(rows).T
    try:
        from IPython.display import display; display(cmp_df)
    except Exception:
        print(cmp_df.to_string())
else:
    cmp_df = pd.DataFrame()
    print("no serving/training/proxy results found yet")


## 3. Cost — $/M-tokens (via `cost_model.py`, if available)

In [ ]:
from argparse import Namespace

cost_df = pd.DataFrame()
if not HAVE_COST:
    print("cost_model.py not importable — skipping cost section. "
          "It ships in the cost-model PR; rerun once it is on main.")
else:
    serving = [d for _, d in RUNS if d.get("schema", "").startswith("vllm-serving-bench")]
    if not serving:
        print("no vLLM serving JSONs to cost")
    else:
        # default assumptions; edit to taste
        args = Namespace(price=0.12, pue=1.4, amortization_years=3.0, util=0.5,
                         server_overhead=1.3, power_watts=None, gpu_capex=None, gpus=None)
        crows = {}
        for d in serving:
            r = CM.analyze(d, args)
            crows[r["label"]] = {
                "model": r["model"],
                "$/M out": round(r["out"]["total"], 4) if r["out"] else None,
                "$/M total-tok": round(r["tot"]["total"], 4) if r["tot"] else None,
                "energy $/M out": round(r["out"]["energy"], 4) if r["out"] else None,
                "hardware $/M out": round(r["out"]["hardware"], 4) if r["out"] else None,
            }
        cost_df = pd.DataFrame(crows).T
        print("Assumptions: $0.12/kWh, PUE 1.4, 3y amortization, util 50%")
        try:
            from IPython.display import display; display(cost_df)
        except Exception:
            print(cost_df.to_string())


## 4. Other workloads — cross-framework, peak, embeddings, image-gen

In [ ]:
def show(title, items, fn):
    if not items:
        return
    print(f"\n#### {title}")
    frames = []
    for p, d in items:
        sub = fn(d)
        if sub is not None and len(sub):
            frames.append(sub)
    if frames:
        out = pd.concat(frames, ignore_index=True)
        try:
            from IPython.display import display; display(out)
        except Exception:
            print(out.to_string(index=False))

def gpu_of(d):
    e = d.get("env", {}); return f"{e.get('gpu_name','?')} ({e.get('platform','?')})"

def f_optimum(d):
    s = d.get("summary", {})
    return pd.DataFrame([{"run": gpu_of(d), "backend": k, **v} for k, v in s.items()])

def f_trtllm(d):
    s = d.get("summary", {})
    return pd.DataFrame([{"run": gpu_of(d), "model": d.get("model"),
                          "concurrency": k, **v} for k, v in s.items()])

def f_list(d):
    return pd.DataFrame([{"run": gpu_of(d), "model": d.get("model"), **r}
                         for r in d.get("results", [])])

show("Cross-framework (optimum-benchmark)", by_schema.get("optimum-bench/1.0", []), f_optimum)
show("Peak ceiling (TensorRT-LLM)", by_schema.get("trtllm-bench/1.0", []), f_trtllm)
show("Embeddings", by_schema.get("embeddings-bench/1.0", []), f_list)
show("Image generation", by_schema.get("image-gen-bench/1.0", []), f_list)


## 5. Charts

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except Exception as e:
    HAVE_PLT = False
    print("matplotlib unavailable, skipping charts:", e)

made = False
# Serving output throughput per platform (max-throughput point)
serv = {CR.label(d): CR.flatten_vllm(d).get("out tok/s (max)")
        for _, d in RUNS if d.get("schema", "").startswith("vllm-serving-bench")}
serv = {k: v for k, v in serv.items() if isinstance(v, (int, float))}
if HAVE_PLT and serv:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(list(serv), list(serv.values()))
    ax.set_ylabel("output tok/s (max)"); ax.set_title("vLLM serving throughput")
    plt.xticks(rotation=20, ha="right"); plt.tight_layout()
    fig.savefig("report_serving_throughput.png", dpi=120); made = True
    try:
        from IPython.display import Image, display; display(Image("report_serving_throughput.png"))
    except Exception:
        print("saved report_serving_throughput.png")

if HAVE_PLT and HAVE_COST and len(cost_df):
    col = "$/M out"
    vals = cost_df[col].dropna()
    if len(vals):
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.bar(vals.index.astype(str), vals.values)
        ax.set_ylabel(col); ax.set_title("Serving cost ($/M output tokens)")
        plt.xticks(rotation=20, ha="right"); plt.tight_layout()
        fig.savefig("report_cost.png", dpi=120); made = True
        try:
            from IPython.display import Image, display; display(Image("report_cost.png"))
        except Exception:
            print("saved report_cost.png")

if not made:
    print("no chartable data yet — run the benchmarks first")


## 6. Save a combined report (Markdown + JSON)

In [ ]:
report = {
    "generated": __import__("datetime").datetime.utcnow().isoformat() + "Z",
    "n_result_files": len(RUNS),
    "schemas": {k: len(v) for k, v in by_schema.items()},
    "comparison": cmp_df.to_dict() if len(cmp_df) else {},
    "cost": cost_df.to_dict() if HAVE_COST and len(cost_df) else {},
}
with open("aiaas_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

def as_table(df):
    try:
        return df.to_markdown()            # needs tabulate
    except Exception:
        return "```\n" + df.to_string() + "\n```"

lines = ["# AIaaS Benchmark Report", "",
         f"- generated: {report['generated']}",
         f"- result files: {report['n_result_files']}",
         f"- schemas: {report['schemas']}", ""]
if len(cmp_df):
    lines += ["## Serving / training / proxy", "", as_table(cmp_df), ""]
if HAVE_COST and len(cost_df):
    lines += ["## Cost ($/M-tokens)", "", as_table(cost_df), ""]
with open("aiaas_report.md", "w") as f:
    f.write("\n".join(lines))
print("Wrote aiaas_report.json and aiaas_report.md")
